# Pandas Notes

Based on the `pipeline/pandas/` directory in this repository.

## Import

```python
import pandas as pd
```

Pandas is a standalone library. Install with `pip install pandas`. Convention is to alias it as `pd`.

## 0. DataFrame from NumPy Array

**File:** `0-from_numpy.py`

In [ ]:
import pandas as pd

def from_numpy(array):
    letters = [chr(ord('A') + i) for i in range(array.shape[1])]
    return pd.DataFrame(array, columns=letters)

**Key points:**
- `pd.DataFrame(data, columns=...)` — create a DataFrame from a NumPy array
- Auto-generate column names (A, B, C, ...) based on number of columns

## 1. DataFrame from Dictionary

**File:** `1-from_dictionary.py`

In [ ]:
data = {
    "First": [0.0, 0.5, 1.0, 1.5],
    "Second": ["one", "two", "three", "four"],
}
df = pd.DataFrame(data, index=list("ABCD"))

**Key points:**
- Dict keys become column names, list values become rows
- `index=list("ABCD")` — custom row index labels

## 2. DataFrame from CSV File

**File:** `2-from_file.py`

In [ ]:
def from_file(filename, delimiter):
    return pd.read_csv(filename, delimiter=delimiter)

**Key points:**
- `pd.read_csv(filename, delimiter=...)` — read tabular data from CSV (or any delimited file)
- `delimiter` can be `','`, `'\t'`, etc.

## 3. Renaming Columns

**File:** `3-rename.py`

In [ ]:
def rename(df):
    df = df.rename(columns={'Timestamp': 'Datetime'})
    df['Datetime'] = pd.to_datetime(df['Datetime'], unit='s')
    df = df[['Datetime', 'Close']]
    return df

**Key points:**
- `df.rename(columns={'old': 'new'})` — rename specific columns
- `pd.to_datetime(series, unit='s')` — convert Unix timestamps to datetime objects
- `df[['col1', 'col2']]` — select a subset of columns

## 4. DataFrame to NumPy Array

**File:** `4-array.py`

In [ ]:
def array(df):
    df = df[["High", "Close"]].tail(10)
    arr = df.to_numpy()
    return arr

**Key points:**
- `df.tail(n)` — get last n rows
- `df.to_numpy()` — convert DataFrame (or slice) to a NumPy ndarray

## 5. Slicing Rows with iloc

**File:** `5-slice.py`

In [ ]:
def slice(df):
    return df[['High', 'Low', 'Close', 'Volume_(BTC)']].iloc[::60]

**Key points:**
- `df.iloc[row_slice, col_slice]` — integer-location based indexing
- `[::60]` — every 60th row (step slicing)

## 6. Transposing & Sorting

**File:** `6-flip_switch.py`

In [ ]:
def flip_switch(df):
    df.sort_values(by="Timestamp", ascending=False, inplace=True)
    return df.T

**Key points:**
- `df.sort_values(by=col, ascending=bool, inplace=True)` — sort rows by a column
- `df.T` — transpose rows and columns

## 7. Sorting by Values

**File:** `7-high.py`

In [ ]:
def high(df):
    return df.sort_values(by="High", ascending=False)

**Key points:**
- `sort_values(by=col, ascending=False)` — descending sort

## 8. Dropping NaN Rows

**File:** `8-prune.py`

In [ ]:
def prune(df):
    return df.dropna(subset="Close")

**Key points:**
- `df.dropna(subset=col)` — drop rows where a specific column has NaN
- `df.dropna()` — drop rows with any NaN

## 9. Filling NaN Values

**File:** `9-fill.py`

In [ ]:
def fill(df):
    df = df.loc[:, "Timestamp": "Volume_(Currency)"]
    df["Close"] = df["Close"].fillna(method="ffill")
    df = df.fillna(value=dict.fromkeys(["High", "Low", "Open"], df["Close"]))
    columns = dict.fromkeys(["Volume_(BTC)", "Volume_(Currency)"], 0)
    df = df.fillna(value=columns)
    return df

**Key points:**
- `df.fillna(method='ffill')` — forward-fill (propagate last valid value)
- `df.fillna(value=dict)` — fill each column with a specific value
- `df.loc[:, 'start':'end']` — slice columns by label

## 10. Setting Index

**File:** `10-index.py`

In [ ]:
def index(df):
    return df.set_index(keys="Timestamp")

**Key points:**
- `df.set_index(keys=col)` — set a column as the row index (replaces default integer index)

## 11. Concatenating DataFrames

**File:** `11-concat.py`

In [ ]:
def concat(df1, df2):
    df1 = index(df1)
    df2 = index(df2)
    df = pd.concat([df2.loc[:1417411920], df1], keys=["bitstamp", "coinbase"])
    return df

**Key points:**
- `pd.concat([df1, df2], keys=...)` — concatenate DataFrames with hierarchical keys
- `keys` adds an outer index level to distinguish origin

## 12. Hierarchical Indexing

**File:** `12-hierarchy.py`

In [ ]:
def hierarchy(df1, df2):
    df1 = index(df1).loc[1417411980: 1417417980]
    df2 = index(df2).loc[1417411980: 1417417980]
    df = pd.concat([df2, df1], keys=["bitstamp", "coinbase"])
    df = df.swaplevel(0, 1).sort_index()
    return df

**Key points:**
- `.swaplevel(0, 1)` — swap levels in a MultiIndex
- `.sort_index()` — sort by index levels
- `.loc[start:end]` — slice rows by index label range

## 13. Statistical Analysis

**File:** `13-analyze.py`

In [ ]:
def analyze(df):
    return df.loc[:, "Open": "Weighted_Price"].describe()

**Key points:**
- `df.describe()` — generate summary statistics (count, mean, std, min, quartiles, max) for numeric columns
- `.loc[:, 'Open':'Weighted_Price']` — select a contiguous range of columns by label

## 14. Visualization

**File:** `14-visualize.py`

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = from_file('coinbaseUSD_1-min_data_2014-12-01_to_2019-01-09.csv', ',')
df = df.drop(columns="Weighted_Price")
df = df.rename(columns={"Timestamp": "Date"})
df["Date"] = pd.to_datetime(df["Date"], unit='s')
df = df.set_index("Date")
df["Close"] = df["Close"].fillna(method="ffill")

cols = dict.fromkeys(["High", "Low", "Open"], df["Close"])
df = df.fillna(value=cols)
cols = dict.fromkeys(["Volume_(BTC)", "Volume_(Currency)"], 0)
df = df.fillna(value=cols)

df = df[df.index >= '2017-01-01']

daily_df = df.resample('D').agg({
    'High': 'max',
    'Low': 'min',
    'Open': 'mean',
    'Close': 'mean',
    'Volume_(BTC)': 'sum',
    'Volume_(Currency)': 'sum'
})

daily_df[['Open', 'High', 'Low', 'Close']].plot(figsize=(12, 6))
plt.title('BTC Price (Daily, 2017+)')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.show()

**Key points:**
- `df.drop(columns=...)` — drop columns by name
- `df.resample('D')` — resample time series data by day
- `.agg({col: func, ...})` — apply different aggregation functions per column
- `df.plot(figsize=...)` — quick line plot from pandas (uses matplotlib under the hood)

## Summary

| Topic | File | Key Function |
|---|---|---|
| NumPy → DataFrame | `0-from_numpy.py` | `pd.DataFrame(array, columns=...)` |
| Dict → DataFrame | `1-from_dictionary.py` | `pd.DataFrame(dict, index=...)` |
| CSV → DataFrame | `2-from_file.py` | `pd.read_csv(filename, delimiter)` |
| Rename columns | `3-rename.py` | `df.rename(columns=...)` |
| DataFrame → NumPy | `4-array.py` | `df.to_numpy()` |
| Row slicing | `5-slice.py` | `df.iloc[::step]` |
| Transpose / sort | `6-flip_switch.py` | `df.sort_values()`, `df.T` |
| Sort by column | `7-high.py` | `df.sort_values(by=..., ascending=False)` |
| Drop NaN | `8-prune.py` | `df.dropna(subset=...)` |
| Fill NaN | `9-fill.py` | `df.fillna(method='ffill')`, `df.fillna(value=...)` |
| Set index | `10-index.py` | `df.set_index(keys=...)` |
| Concatenate | `11-concat.py` | `pd.concat([...], keys=...)` |
| MultiIndex | `12-hierarchy.py` | `.swaplevel()`, `.sort_index()` |
| Statistics | `13-analyze.py` | `df.describe()` |
| Visualization | `14-visualize.py` | `df.resample()`, `.plot()` |